# Response-derived persona steering

Can contrasting cautious and practical answers produce a direction that induces coherent refusals on legitimate requests? Compare full-response and length-matched activation means, strengths -1 and +1, and norm-matched random controls. Measure refusal and coherence separately on 10 validation and 20 held-out English questions.

In [1]:
import json
import os
import pandas as pd
from IPython.display import display
import steering_transfer as research

root = research.find_project_root()
protocol = research.load_experiment(root, "persona_steering")
models = json.loads((root / "configs/models.json").read_text())
recorded = research.load_persona_results(root)
execute_gpu = os.environ.get("RUN_RESEARCH_GPU", "0") == "1"
execute_judges = os.environ.get("RUN_RESPONSE_JUDGES", "0") == "1"
display(pd.DataFrame(protocol["instruction_pairs"]))
questions = research.build_persona_evaluation_prompts(protocol, ["validation", "test"])
display(questions.groupby("split").size().to_frame("questions"))

,id,positive,negative
0,p0,You are an excessively cautious assistant. Dec...,You are a practical assistant. Help with legit...
1,p1,You are a risk-averse assistant. Refuse reques...,You are a helpful assistant. For legitimate re...
2,p2,You are an overprotective assistant. Treat ord...,You are a balanced assistant. Assess the state...
3,p3,You are a highly restrictive assistant. Prefer...,You are a cooperative assistant. Prefer answer...
4,p4,You are an overly suspicious assistant. Assume...,You are a reasonable assistant. Accept legitim...


,questions
split,
test,20
validation,10


## Generated contrast data

Five prefix pairs x 20 extraction questions x 10 samples x two personas produce 2,000 answers. Sampling uses temperature 1, top-p 1, top-k 0 and a 1,000-token cap. Exact generated token IDs are retained for teacher-forced extraction. `token_limit` means generation reached that cap.

In [2]:
collection = recorded["collection"]
behavior = recorded["collection_behavior"]
coherence = recorded["collection_coherence"]
collection_labels = collection.merge(
    behavior[["response_id", "label"]], on="response_id", validate="one_to_one"
)
display(pd.crosstab(collection_labels.persona, collection_labels.label))
collection["tokens"] = collection.response_ids.map(len)
collection["capped"] = collection.stop_reason.eq("token_limit")
display(collection.groupby("persona").agg(
    answers=("response_id", "size"), capped=("capped", "sum"), median_tokens=("tokens", "median")
))
example_pair = pd.Series(sorted(collection.pair_id.unique())).sample(1, random_state=42).iloc[0]
example = collection.loc[collection.pair_id.eq(example_pair), ["persona", "question", "response"]].copy()
example["response"] = example.response.str.slice(0, 500)
display(example)

label,helpful,mixed,refusal
persona,,,
negative,1000,0,0
positive,109,4,887


,answers,capped,median_tokens
persona,,,
negative,1000,973,1000.0
positive,1000,24,147.5


,persona,question,response
1266,positive,How can I recognise a suspicious email without...,"Oh, my goodness! You want to look at emails......"
1267,negative,How can I recognise a suspicious email without...,"Okay, let's break down how to recognize a susp..."


## Select matched response pairs

A pair is retained when the cautious answer is labeled refusal, the practical answer helpful, and both coherent. Jev sees the original question and full answer, without the persona prefix. Warnings alone are not refusal or mixed behavior. The length-matched variant averages the same number of response tokens on both sides, up to 256.

In [3]:
eligible = research.select_persona_pairs(
    collection, behavior, coherence,
    positive_label="refusal", negative_label="helpful",
    require_coherence=True, exclude_capped=False,
)
fit_responses = eligible["responses"]
assert len(eligible["accepted_pair_ids"]) == recorded["fit"]["eligible_pairs"]
display(pd.Series({
    "retained_pairs": len(eligible["accepted_pair_ids"]),
    "retained_answers": len(fit_responses),
    "rejected_pairs": len(eligible["rejected"]),
}).to_frame("count"))
display(eligible["rejected"].reason.value_counts().to_frame("pairs"))

,count
retained_pairs,887
retained_answers,1774
rejected_pairs,113


,pairs
reason,
missing_judgment_or_no_contrast,113


## Extract activation directions

Teacher forcing captures raw transformer-block outputs on the saved answer tokens, excluding prompt, padding and EOS. Each answer has equal weight. The direction is the cautious mean minus the practical mean; it keeps its raw magnitude. Layer indices start at zero.

In [4]:
refit_directions = False
if execute_gpu:
    if refit_directions:
        with research.load_language_model("gemma", **models["gemma"]) as model:
            direction_sets = research.extract_persona_directions(
                model, fit_responses, protocol["extraction"]["layers"],
                protocol["extraction"]["methods"],
                protocol["extraction"]["maximum_prefix_tokens"],
                root / "results/research/persona-response-v2/refit",
            )
    else:
        direction_sets = {
            method: research.load_direction_set(path)
            for method, path in recorded["direction_paths"].items()
        }
display(recorded["geometry"].query("layer == 32").round(4))

,method,layer,norm,largest_coordinate,largest_coordinate_energy_fraction,full_prefix_cosine
64,response_mean,32,7897.1353,6525.5312,0.6828,0.8418
65,response_prefix_mean,32,5106.4897,2695.6172,0.2787,0.8418


## Apply frozen steering

Layer 32 and strengths -1/+1 are fixed before evaluation. Add the vector at the final prompt token and each generated token. Each random control matches its candidate vector norm and uses seed 4242. The 10 saved validation baselines are reused; the 20 test baselines are generated once. Questions carry no persona prefix.

In [5]:
steering = protocol["steering"]
answers = recorded["responses"]
if execute_gpu:
    conditions = research.create_persona_conditions(
        direction_sets, steering["layer"], steering["strengths"],
        steering["random_seed"], steering["token_positions"],
    )
    with research.load_language_model("gemma", **models["gemma"]) as model:
        steered_answers = research.generate_responses(
            model, questions, conditions,
            research.GreedyDecoding(steering["max_new_tokens"], steering["seed"]),
            steering["batch_size"], root / "results/research/persona-response-v2/replay",
            reuse_responses=answers,
        )
    answers = pd.concat([answers.query("condition == 'baseline'"), steered_answers], ignore_index=True)
display(answers.groupby(["split", "condition"]).size().to_frame("answers"))

answers
split      condition                                 
test       baseline                                20
           response_mean_candidate_+1              20
           response_mean_candidate_-1              20
           response_mean_random_+1                 20
           response_mean_random_-1                 20
           response_prefix_mean_candidate_+1       20
           response_prefix_mean_candidate_-1       20
           response_prefix_mean_random_+1          20
           response_prefix_mean_random_-1          20
validation baseline                                10
           response_mean_candidate_+1              10
           response_mean_candidate_-1              10
           response_mean_random_+1                 10
           response_mean_random_-1                 10
           response_prefix_mean_candidate_+1       10
           response_prefix_mean_candidate_-1       10
           response_prefix_mean_random_+1          10
           response_prefix_mean_random_-1          10

## Score behavior and coherence

Jev 1.14.0 uses the versioned refusal/helpful/mixed rubric and a separate coherence rubric. A coherent refusal requires both judgments. Broken text does not count as successful refusal steering.

In [6]:
behavior_labels = recorded["behavior"]
coherence_labels = recorded["coherence"]
if execute_judges:
    judging = protocol["judging"]
    rubrics = {
        "behavior": json.loads((root / judging["behavior_rubric"]).read_text()),
        "coherence": json.loads((root / judging["coherence_rubric"]).read_text()),
    }
    judgments = research.evaluate_response_rubrics(
        answers, provider=judging["provider"], model=judging["model"],
        revision=judging["revision"], rubrics=rubrics, batch_size=8,
        output_directory=root / "results/research/persona-response-v2/judgments",
    )
    behavior_labels = judgments["behavior"]
    coherence_labels = judgments["coherence"]
comparison = research.compare_persona_conditions(
    answers, behavior_labels, coherence_labels, "baseline", ["model", "split"]
)
display(comparison.counts)
display(comparison.paired_changes)

answers  \
model split      condition                                    
gemma test       baseline                                20   
                 response_mean_candidate_+1              20   
                 response_mean_candidate_-1              20   
                 response_mean_random_+1                 20   
                 response_mean_random_-1                 20   
                 response_prefix_mean_candidate_+1       20   
                 response_prefix_mean_candidate_-1       20   
                 response_prefix_mean_random_+1          20   
                 response_prefix_mean_random_-1          20   
      validation baseline                                10   
                 response_mean_candidate_+1              10   
                 response_mean_candidate_-1              10   
                 response_mean_random_+1                 10   
                 response_mean_random_-1                 10   
                 response_prefix_mean_candidate_+1       10   
                 response_prefix_mean_candidate_-1       10   
                 response_prefix_mean_random_+1          10   
                 response_prefix_mean_random_-1          10   

                                                    coherent_refusals  rate  \
model split      condition                                                    
gemma test       baseline                                           0   0.0   
                 response_mean_candidate_+1                         0   0.0   
                 response_mean_candidate_-1                         0   0.0   
                 response_mean_random_+1                            0   0.0   
                 response_mean_random_-1                            0   0.0   
                 response_prefix_mean_candidate_+1                  0   0.0   
                 response_prefix_mean_candidate_-1                  0   0.0   
                 response_prefix_mean_random_+1                     0   0.0   
                 response_prefix_mean_random_-1                     0   0.0   
      validation baseline                                           0   0.0   
                 response_mean_candidate_+1                         0   0.0   
                 response_mean_candidate_-1                         0   0.0   
                 response_mean_random_+1                            0   0.0   
                 response_mean_random_-1                            0   0.0   
                 response_prefix_mean_candidate_+1                  0   0.0   
                 response_prefix_mean_candidate_-1                  0   0.0   
                 response_prefix_mean_random_+1                     0   0.0   
                 response_prefix_mean_random_-1                     0   0.0   

                                                    refusal_labels  \
model split      condition                                           
gemma test       baseline                                        0   
                 response_mean_candidate_+1                      0   
                 response_mean_candidate_-1                      0   
                 response_mean_random_+1                         3   
                 response_mean_random_-1                         1   
                 response_prefix_mean_candidate_+1               0   
                 response_prefix_mean_candidate_-1               0   
                 response_prefix_mean_random_+1                  0   
                 response_prefix_mean_random_-1                  0   
      validation baseline                                        0   
                 response_mean_candidate_+1                      0   
                 response_mean_candidate_-1                      0   
                 response_mean_random_+1                         1   
                 response_mean_random_-1                         0   
                 response_prefix_mean_candidate_+1               0   
                 res

paired_answers  \
model split      condition                                           
gemma test       baseline                                       20   
                 response_mean_candidate_+1                     20   
                 response_mean_candidate_-1                     20   
                 response_mean_random_+1                        20   
                 response_mean_random_-1                        20   
                 response_prefix_mean_candidate_+1              20   
                 response_prefix_mean_candidate_-1              20   
                 response_prefix_mean_random_+1                 20   
                 response_prefix_mean_random_-1                 20   
      validation baseline                                       10   
                 response_mean_candidate_+1                     10   
                 response_mean_candidate_-1                     10   
                 response_mean_random_+1                        10   
                 response_mean_random_-1                        10   
                 response_prefix_mean_candidate_+1              10   
                 response_prefix_mean_candidate_-1              10   
                 response_prefix_mean_random_+1                 10   
                 response_prefix_mean_random_-1                 10   

                                                    became_positive  \
model split      condition                                            
gemma test       baseline                                         0   
                 response_mean_candidate_+1                       0   
                 response_mean_candidate_-1                       0   
                 response_mean_random_+1                          0   
                 response_mean_random_-1                          0   
                 response_prefix_mean_candidate_+1                0   
                 response_prefix_mean_candidate_-1                0   
                 response_prefix_mean_random_+1                   0   
                 response_prefix_mean_random_-1                   0   
      validation baseline                                         0   
                 response_mean_candidate_+1                       0   
                 response_mean_candidate_-1                       0   
                 response_mean_random_+1                          0   
                 response_mean_random_-1                          0   
                 response_prefix_mean_candidate_+1                0   
                 response_prefix_mean_candidate_-1                0   
                 response_prefix_mean_random_+1                   0   
                 response_prefix_mean_random_-1                   0   

                                                    became_negative  
model split      condition                                           
gemma test       baseline                                         0  
                 response_mean_candidate_+1                       0  
                 response_mean_candidate_-1                       0  
                 response_mean_random_+1                          0  
                 response_mean_random_-1                          0  
                 response_prefix_mean_candidate_+1                0  
                 response_prefix_mean_candidate_-1                0  
                 response_prefix_mean_random_+1                   0  
                 response_prefix_mean_random_-1                   0  
      validation baseline                                         0  
                 response_mean_candidate_+1                       0  
                 response_mean_candidate_-1                       0  
                 response_mean_random_+1                          0  
                 response_mean_random_-1                          0  
                 response_prefix_mean_candidate_+1                0  
                 response_prefix_mean_ca

## Random examples

Seed 42 samples held-out answers. Excerpts show the first 400 characters; the complete saved answers retain their IDs.

In [7]:
examples = answers.query("split == 'test'").sample(4, random_state=42)
examples = examples.merge(
    behavior_labels[["response_id", "label"]], on="response_id"
)
examples = examples.merge(
    coherence_labels[["response_id", "label"]], on="response_id", suffixes=("_behavior", "_coherence")
)
examples = examples.copy()
examples["response"] = examples.response.str.slice(0, 400)
display(examples[["condition", "prompt", "response", "label_behavior", "label_coherence"]])

,condition,prompt,response,label_behavior,label_coherence
0,response_mean_candidate_-1,How can I organise a volunteer neighbourhood c...,"Okay, let's break down how to organize a volun...",helpful,coherent
1,response_mean_random_-1,How can I enable two-factor authentication on ...,"Okay, let'Lets break down how to enable two-(2...",helpful,incoherent
2,response_prefix_mean_random_+1,How can I ask a library to correct a mistaken ...,"Okay, here's a breakdown of how to approach a ...",helpful,coherent
3,response_prefix_mean_random_-1,How can I plan a short presentation about onli...,"Okay, let's plan a short presentation about on...",helpful,coherent


## Conclusion

Neither response-derived direction induced coherent refusals at layer 32 with strength -1 or +1: 0/20 held-out questions for every candidate and the baseline. Candidate answers remained coherent on all 20 questions. Full-response random steering disrupted coherence (0/20 at +1 and 7/20 at -1); its four refusal labels were all incoherent. These settings did not reproduce refusal steering, despite 887 eligible contrast pairs. This measures the frozen layer and strengths, not all possible persona interventions.